In [0]:
-- Game - Shot Log
-- Every shot attempt in one game with location, for a shot map.
-- x_ft / y_ft are flipped so the shooting team always attacks toward the net at x = +89.
-- Blocked shots are recorded where they were blocked, not where they were taken.
-- Change the values in params only; everything below reads from it.
WITH params AS (                              -- change settings here only
  SELECT
    2025020001 AS game_id,          -- e.g. 2025020001 (season 2025, 02 = regular season, game 0001); see Game - Find Game ID
    CAST(NULL AS STRING) AS team    -- team abbreviation, e.g. 'MTL'; NULL = both teams
),
game_players AS (                        -- everyone dressed in the game (game rosters)
  SELECT r.game_id, r.player_id, concat_ws(' ', r.first_name, r.last_name) AS full_name,
         r.position_code, r.sweater_number, r.team_id, t.abbrev AS team,
         r.team_id = g.home_team_id AS is_home, g.home_team_id
  FROM nhl.silver.game_rosters r
  JOIN nhl.gold.dim_game g ON r.game_id = g.game_id
  JOIN nhl.gold.dim_team t ON r.team_id = t.team_id
  CROSS JOIN params pr
  WHERE r.game_id = pr.game_id
),
plays AS (                               -- every play in the game, with the owning team's strength
  SELECT t.abbrev AS team,
         f.event_owner_team_id = g.home_team_id AS owner_is_home,
         IF(f.event_owner_team_id = g.home_team_id, f.situation_code, reverse(f.situation_code)) AS owner_situation,
         CASE WHEN substr(owner_situation, 1, 1) = '0' THEN 'EN'      -- opponent net empty
              WHEN substr(owner_situation, 3, 1) > substr(owner_situation, 2, 1)
               AND substr(owner_situation, 4, 1) = '1' THEN 'PP'      -- more skaters, own goalie in
              WHEN substr(owner_situation, 3, 1) < substr(owner_situation, 2, 1) THEN 'SH'
              ELSE 'EV' END AS strength,
         CASE WHEN f.period_type IN ('OT', 'SO') THEN f.period_type
              ELSE CAST(f.period_number AS STRING) END AS period,
         SQRT(POW(89 - ABS(f.x_coord), 2) + POW(f.y_coord, 2)) AS distance_ft,
         f.*
  FROM nhl.gold.fact_event f
  JOIN nhl.gold.dim_game g ON f.game_id = g.game_id
  LEFT JOIN nhl.gold.dim_team t ON t.team_id = f.event_owner_team_id
  CROSS JOIN params pr
  WHERE f.game_id = pr.game_id
)
SELECT p.period,
       p.time_in_period          AS time,
       p.team,
       sh.full_name              AS shooter,
       CASE p.type_desc_key WHEN 'goal' THEN 'Goal' WHEN 'shot-on-goal' THEN 'Saved'
                            WHEN 'missed-shot' THEN 'Missed' ELSE 'Blocked' END AS result,
       p.shot_type,
       p.strength,
       ROUND(p.distance_ft, 0)   AS distance_ft,
       IF(p.attacks_right, p.x_coord, -p.x_coord) AS x_ft,
       IF(p.attacks_right, p.y_coord, -p.y_coord) AS y_ft,
       gl.full_name              AS goalie,
       bl.full_name              AS blocked_by
FROM (SELECT *,                           -- home defends left => home attacks right; older games lack the side
             COALESCE(owner_is_home = (home_team_defending_side = 'left'), x_coord >= 0) AS attacks_right
      FROM plays) p
LEFT JOIN game_players sh ON sh.player_id = COALESCE(p.scoring_player_id, p.shooting_player_id)
LEFT JOIN game_players gl ON gl.player_id = p.goalie_in_net_id
LEFT JOIN game_players bl ON bl.player_id = p.blocking_player_id
CROSS JOIN params pr
WHERE p.type_desc_key IN ('goal', 'shot-on-goal', 'missed-shot', 'blocked-shot')
  AND COALESCE(p.period_type, '') <> 'SO'    -- shootout attempts aren't stats
  AND (pr.team IS NULL OR p.team = pr.team)
ORDER BY p.period_number, p.sort_order;